<center><h1>Zheng_Yifei_HW3</h1></center>
<br>
<br>

Name: Yifei Zheng
<br>
Github Username: yyl-ff
<br>
USC ID: 8538054131

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [5]:
import os
import glob
import numpy as np
import pandas as pd

from scipy.stats import bootstrap
from sklearn.utils import resample

Get the AReM Data Set

In [7]:
data_path = "AReM"

activity_folders = sorted(os.listdir(data_path))

print("Dataset folder:", data_path)
print("Activity folders:", activity_folders)

Dataset folder: AReM
Activity folders: ['.DS_Store', 'bending1', 'bending2', 'bendingType.pdf', 'cycling', 'lying', 'sensorsPlacement.pdf', 'sitting', 'standing', 'walking']


### (b) Test and Train Data

In [9]:
train_files = []
test_files = []

for activity in activity_folders:
    folder_path = os.path.join(data_path, activity)
    csv_files = glob.glob(os.path.join(folder_path, "dataset*.csv"))

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        dataset_number = int(file_name.replace("dataset", "").replace(".csv", ""))

        if activity in ["bending1", "bending2"]:
            if dataset_number in [1, 2]:
                test_files.append((activity, file_path))
            else:
                train_files.append((activity, file_path))
        else:
            if dataset_number in [1, 2, 3]:
                test_files.append((activity, file_path))
            else:
                train_files.append((activity, file_path))

print("Number of training instances:", len(train_files))
print("Number of testing instances:", len(test_files))
print("Total number of instances:", len(train_files) + len(test_files))

Number of training instances: 69
Number of testing instances: 19
Total number of instances: 88


### (c) Feature Extraction

#### i. Research

Common time-domain features used in time series classification include minimum, maximum, mean, median, standard deviation, variance, range, first quartile, third quartile, interquartile range, root mean square, skewness, kurtosis, and zero-crossing rate. These features describe the distribution, variability, and shape of a time series.

#### ii. Extraction

In [27]:
def read_time_series(file_path):
    # bending2/dataset4.csv uses spaces instead of commas
    if file_path.endswith(os.path.join("bending2", "dataset4.csv")):
        data = pd.read_csv(
            file_path,
            skiprows=5,
            sep=r"\s+",
            header=None,
            usecols=range(7),
            engine="python"
        )
    else:
        data = pd.read_csv(
            file_path,
            skiprows=5,
            header=None,
            usecols=range(7),
            engine="python"
        )

    # Remove the first column because it represents time
    time_series = data.iloc[:, 1:7].apply(
        pd.to_numeric,
        errors="coerce"
    )

    return time_series.dropna()


def extract_features(file_list, data_type):
    rows = []

    for activity, file_path in file_list:
        time_series = read_time_series(file_path)

        row = {
            "Activity": activity,
            "File": os.path.basename(file_path),
            "Set": data_type
        }

        for i in range(6):
            values = time_series.iloc[:, i]

            row[f"min{i + 1}"] = values.min()
            row[f"max{i + 1}"] = values.max()
            row[f"mean{i + 1}"] = values.mean()
            row[f"median{i + 1}"] = values.median()
            row[f"std{i + 1}"] = values.std()
            row[f"1st_quart{i + 1}"] = values.quantile(0.25)
            row[f"3rd_quart{i + 1}"] = values.quantile(0.75)

        rows.append(row)

    return pd.DataFrame(rows)


# Extract features from the training and test instances
train_features = extract_features(train_files, "Train")
test_features = extract_features(test_files, "Test")

# Combine all 88 instances
feature_data = pd.concat(
    [train_features, test_features],
    ignore_index=True
)

# Add the instance number
feature_data.insert(
    0,
    "Instance",
    range(1, len(feature_data) + 1)
)

# Identify the 42 extracted feature columns
feature_columns = [
    column for column in feature_data.columns
    if column not in ["Instance", "Activity", "File", "Set"]
]

print("Number of instances:", feature_data.shape[0])
print("Number of extracted features:", len(feature_columns))
print(
    "Total missing values:",
    feature_data[feature_columns].isna().sum().sum()
)

display(
    feature_data.head().style.hide(axis="index")
)

Number of instances: 88
Number of extracted features: 42
Total missing values: 0


Instance,Activity,File,Set,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
1,bending1,dataset7.csv,Train,36.250000,48.000000,43.969125,44.500000,1.618364,43.310000,44.670000,0.000000,1.500000,0.413125,0.470000,0.263111,0.430000,0.500000,1.500000,26.330000,15.868021,16.250000,3.742420,14.250000,18.000000,0.000000,5.170000,0.666354,0.470000,0.788985,0.000000,0.940000,11.330000,30.750000,22.103750,21.670000,3.318301,20.500000,23.750000,0.000000,2.960000,0.555312,0.490000,0.487826,0.000000,0.830000
2,bending1,dataset6.csv,Train,37.000000,48.000000,43.454958,43.250000,1.386098,42.500000,45.000000,0.000000,1.580000,0.378083,0.470000,0.315566,0.000000,0.500000,5.750000,27.000000,15.793333,15.000000,3.847638,13.000000,18.270000,0.000000,10.030000,0.849354,0.500000,0.995761,0.430000,1.120000,8.000000,33.500000,23.034792,23.500000,2.488862,22.250000,24.000000,0.000000,5.260000,0.679646,0.500000,0.622534,0.430000,0.870000
3,bending1,dataset4.csv,Train,33.000000,47.750000,42.179812,43.500000,3.670666,39.150000,45.000000,0.000000,3.000000,0.696042,0.500000,0.630860,0.000000,1.120000,8.500000,30.000000,22.183625,23.000000,3.810469,20.500000,24.372500,0.000000,5.150000,0.989917,0.830000,0.953730,0.430000,1.300000,20.000000,38.670000,33.493917,35.000000,3.849448,30.457500,36.330000,0.000000,2.180000,0.613521,0.500000,0.524317,0.000000,1.000000
4,bending1,dataset5.csv,Train,33.000000,45.750000,41.678063,41.750000,2.243490,41.330000,42.750000,0.000000,2.830000,0.535979,0.500000,0.405469,0.430000,0.710000,3.000000,28.250000,19.006563,19.125000,4.087107,16.500000,22.062500,0.000000,6.420000,0.841875,0.500000,0.928801,0.430000,1.120000,23.670000,37.500000,29.857083,30.000000,2.411026,28.457500,31.250000,0.000000,1.790000,0.383292,0.430000,0.389164,0.000000,0.500000
5,bending1,dataset3.csv,Train,35.000000,47.400000,43.954500,44.330000,1.558835,43.000000,45.000000,0.000000,1.700000,0.426250,0.470000,0.338690,0.000000,0.500000,6.500000,29.750000,22.122354,23.000000,3.030943,19.750000,24.000000,0.000000,4.440000,0.497312,0.430000,0.550657,0.000000,0.830000,29.000000,38.500000,35.588458,36.000000,1.999604,35.362500,36.500000,0.000000,1.790000,0.493292,0.430000,0.513506,0.000000,0.940000


#### iii. Standard Deviation

In [29]:
feature_columns = [
    column for column in feature_data.columns
    if column not in ["Instance", "Activity", "File", "Set"]
]


def sample_std(data, axis=-1):
    return np.std(data, axis=axis, ddof=1)


standard_deviation_results = []

for feature in feature_columns:
    values = feature_data[feature].to_numpy()

    estimated_std = values.std(ddof=1)

    bootstrap_result = bootstrap(
        (values,),
        sample_std,
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        random_state=42
    )

    standard_deviation_results.append({
        "Feature": feature,
        "Standard Deviation": estimated_std,
        "90% CI Lower": bootstrap_result.confidence_interval.low,
        "90% CI Upper": bootstrap_result.confidence_interval.high
    })


std_results = pd.DataFrame(standard_deviation_results)

display(
    std_results.round(4).style.hide(axis="index")
)

Feature,Standard Deviation,90% CI Lower,90% CI Upper
min1,9.570000,8.268800,10.769600
max1,4.394400,3.352100,5.292300
mean1,5.335700,4.705300,5.873900
median1,5.440100,4.784600,5.990800
std1,1.772200,1.569800,1.952900
1st_quart1,6.153600,5.569200,6.639200
3rd_quart1,5.138900,4.331000,5.841200
min2,0.000000,0.000000,0.000000
max2,5.062700,4.627300,5.404200
mean2,1.574200,1.396700,1.704300


#### iv. Select Features

I selected the minimum, mean, and maximum as the three most important time-domain features. The minimum and maximum describe the range of values observed in each time series, while the mean represents its overall level. Together, these features provide a simple summary of the distribution and can help distinguish different human activities.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression is expected to have a training RSS that is less than or equal to that of the linear regression. The cubic model is more flexible and includes the linear model as a special case when the coefficients of \(X^2\) and \(X^3\) are zero. Therefore, adding these terms cannot increase the training RSS and may reduce it by fitting some of the random noise in the training data.

### (b) Linear Test

The linear regression is expected to have a lower test RSS. Since the true relationship is linear, the extra terms in the cubic model are not needed. They may fit noise in the training data and make the cubic model perform worse on new data.

### (c) Not Linear Train

The cubic regression is expected to have a lower training RSS. It is more flexible than the linear regression and can fit nonlinear patterns in the data. Also, adding the squared and cubic terms cannot increase the training RSS.

### (d) Not Linear Testing

There is not enough information to decide which model will have a lower test RSS. If the true relationship is close to cubic, the cubic regression may perform better. If it is only slightly nonlinear or has a different shape, the linear regression may perform better because it is less likely to overfit.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 